# Notebook 03: Gold SCD2 dimensions

## Runtime contract

- **Inputs/runtime:** complete current-state Silver masters; Delta-enabled Fabric Spark.
- **Parameters:** `run_id` correlates the run; small conformed dimensions deliberately scan their complete Silver sources. `effective_dt` optionally overrides the business change date (yyyy-MM-dd).
- **Grain:** source natural key plus `EFFECTIVE_FROM`; CSKS/CEPC identities include `DATBI` so SAP validity intervals are never collapsed.
- **Changes:** business-only hashes exclude every extraction/audit field. First-seen keys cover their SAP `DATAB` (otherwise 1900-01-01), enabling late historical postings.
- **Replay/history:** one atomic MERGE expires changes/removals and inserts versions. Same-day corrections reuse the existing version/SK; unchanged versions are not written. Removed master rows are expired, never physically removed, retaining historical fact references.
- **Dates:** posting lookups intersect SCD2 dates with SAP `DATAB`/`DATBI`. A same-day/future source removal may retain an empty expired validity window rather than incorrectly resolve new facts.
- **Outputs:** existing Gold dimension/date/fiscal-period schemas and `audit.gold_scd2_run_log`; invalid source keys, overlapping SAP intervals, or ambiguous versions fail explicitly.

- **Native CEPC compatibility:** `BUKRS` is optional and emitted as typed null when absent; profit-center identity and joins use client/controlling area/profit center/validity, never that synthetic field.


In [ ]:
run_id = ""
effective_dt = ""  # Optional yyyy-MM-dd business change-date override.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
from functools import reduce
import datetime
import uuid

if not isinstance(run_id, str):
    raise TypeError("run_id must be a string")
run_id = run_id.strip()
if effective_dt.strip():
    datetime.date.fromisoformat(effective_dt.strip())
BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
SLV = f"{BRZ_SLV_LH}.slv"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
HIGH_DATE = "9999-12-31"
BATCH_ID = run_id or str(uuid.uuid4())
RUN_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)


In [ ]:
DIM_CONFIG = {
    "company": {"source": "t001", "keys": ["MANDT", "BUKRS"], "columns": ["MANDT", "BUKRS", "BUTXT", "ORT01", "LAND1", "WAERS", "KTOPL"]},
    "cost_center": {"source": "csks", "keys": ["MANDT", "KOKRS", "KOSTL", "DATBI"], "columns": ["MANDT", "KOKRS", "KOSTL", "BUKRS", "PRCTR", "VERAK", "KOSAR", "KHINR", "DATAB", "DATBI"]},
    "profit_center": {"source": "cepc", "keys": ["MANDT", "KOKRS", "PRCTR", "DATBI"], "optional_columns": {"BUKRS": "string"}, "columns": ["MANDT", "KOKRS", "PRCTR", "BUKRS", "VERAK", "KHINR", "SEGMENT", "DATAB", "DATBI"]},
    "vendor": {"source": "lfa1", "keys": ["MANDT", "LIFNR"], "columns": ["MANDT", "LIFNR", "NAME1", "NAME2", "LAND1", "ORT01", "REGIO", "KTOKK", "STCD1"]},
    "customer": {"source": "kna1", "keys": ["MANDT", "KUNNR"], "columns": ["MANDT", "KUNNR", "NAME1", "NAME2", "LAND1", "ORT01", "REGIO", "KTOKD", "STCD1"]},
    "gl_account": {"source": "gl_account_conformed", "keys": ["MANDT", "KTOPL", "SAKNR"], "columns": ["MANDT", "KTOPL", "SAKNR", "TXT20", "TXT50", "XBILK", "GVTYP", "KTOKS"]},
    "division": {"source": "division_conformed", "keys": ["DIVISION_CODE"], "columns": ["DIVISION_CODE", "DIVISION_NAME", "DATAB", "DATBI"]},
}
EXTRACTION_METADATA = {
    "ROW_HASH", "RECORD_HASH", "EXTRACTION_MODE", "PARTITION_ID", "SOURCE_SCHEMA", "SOURCE_TABLE",
    "EXTRACTED_AT_UTC", "CHECKPOINT_START", "CHECKPOINT_END", "BATCH_RUN_ID", "EXTRACT_DATE",
    "LOAD_DATE", "SOURCE_SYSTEM", "SOURCE_FILE_NAME", "RUN_ID", "WORK_ID", "GOLD_BATCH_ID", "GOLD_LOAD_TIMESTAMP",
}
SOURCE_METADATA = {"BATCH_RUN_ID": "string", "EXTRACT_DATE": "date", "LOAD_DATE": "date", "SOURCE_SYSTEM": "string", "SOURCE_FILE_NAME": "string"}


def business_hash_columns(source_columns: list[str], configured: list[str]) -> list[str]:
    return [column for column in configured if column in source_columns and column.upper() not in EXTRACTION_METADATA]


def sha256_all(columns):
    # Preserve the established business hash algorithm; metadata additions cannot churn versions.
    return F.sha2(F.concat_ws("||", *[
        F.coalesce(F.trim(F.col(column).cast("string")), F.lit("__NULL__")) for column in columns
    ]), 256)


def latest_per_key(df, keys):
    order = [F.col(column).desc_nulls_last() for column in
             ["EXTRACT_DATE", "LOAD_DATE", "SOURCE_FILE_NAME", "BATCH_RUN_ID"] if column in df.columns]
    return (df.withColumn("_RN", F.row_number().over(Window.partitionBy(*keys).orderBy(*(order or [F.lit(1)]))))
            .filter(F.col("_RN") == 1).drop("_RN"))


def dimension_version_start(source_start, has_history: bool, change_date, override: str = ""):
    """Date-grain SCD2: first-seen keys must support late historical facts."""
    if override:
        return datetime.date.fromisoformat(override)
    if has_history:
        return change_date
    return source_start or datetime.date(1900, 1, 1)


def assert_unique(df, keys, label):
    if df.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{label}: duplicate keys {keys}")


def expire_versions(rows, end_date):
    return (rows.withColumn("IS_CURRENT", F.lit(False)).withColumn("EFFECTIVE_TO", end_date)
            .withColumn("GOLD_BATCH_ID", F.lit(BATCH_ID)).withColumn("GOLD_LOAD_TIMESTAMP", F.lit(RUN_TIMESTAMP)))


def scd2_load(dimension_name, cfg):
    source = spark.table(f"{SLV}.{cfg['source']}")
    # CEPC.BUKRS is a synthetic compatibility column, not a native SAP identity.
    for column, data_type in cfg.get("optional_columns", {}).items():
        if column not in source.columns:
            source = source.withColumn(column, F.lit(None).cast(data_type))
    keys = cfg["keys"]
    missing = [key for key in keys if key not in source.columns]
    if missing:
        raise ValueError(f"{dimension_name}: missing business keys {missing}")
    business_columns = business_hash_columns(source.columns, cfg["columns"])
    assert_unique(source, keys, f"{dimension_name} Silver")
    staging = source.select(*business_columns, *[
        (F.col(column).cast(data_type) if column in source.columns else F.lit(None).cast(data_type)).alias(column)
        for column, data_type in SOURCE_METADATA.items()
    ]).withColumn("RECORD_HASH", sha256_all(business_columns))
    null_key = reduce(lambda a, b: a | b, [F.col(key).isNull() for key in keys])
    if staging.filter(null_key).limit(1).count():
        raise ValueError(f"{dimension_name}: null natural key")
    if "DATAB" in business_columns:
        bad_dates = F.col("DATAB").isNull() | F.col("DATBI").isNull() | (F.col("DATAB") > F.col("DATBI"))
        if staging.filter(bad_dates).limit(1).count():
            raise ValueError(f"{dimension_name}: invalid SAP validity interval")
    if dimension_name in {"cost_center", "profit_center"}:
        natural_keys = [key for key in keys if key != "DATBI"]
        interval_order = Window.partitionBy(*natural_keys).orderBy("DATAB", "DATBI").rowsBetween(Window.unboundedPreceding, -1)
        overlaps = staging.withColumn("_PREVIOUS_END", F.max("DATBI").over(interval_order))
        if overlaps.filter(F.col("DATAB") <= F.col("_PREVIOUS_END")).limit(1).count():
            raise ValueError(f"{dimension_name}: overlapping SAP validity intervals")

    target = f"{DIM}.{dimension_name}"
    change_date = datetime.date.fromisoformat(effective_dt.strip()) if effective_dt.strip() else RUN_TIMESTAMP.date()
    if not spark.catalog.tableExists(target):
        empty = (staging.limit(0).withColumn("EFFECTIVE_FROM", F.lit("1900-01-01").cast("date"))
                 .withColumn("EFFECTIVE_TO", F.lit(HIGH_DATE).cast("date")).withColumn("IS_CURRENT", F.lit(True))
                 .withColumn("DIMENSION_SK", F.lit(None).cast("string")).withColumn("GOLD_BATCH_ID", F.lit(BATCH_ID))
                 .withColumn("GOLD_LOAD_TIMESTAMP", F.lit(RUN_TIMESTAMP)))
        empty.write.format("delta").mode("overwrite").saveAsTable(target)
    history = spark.table(target)
    assert_unique(history, ["DIMENSION_SK"], f"{dimension_name} Gold")
    assert_unique(history, keys + ["EFFECTIVE_FROM"], f"{dimension_name} versions")
    current = history.filter(F.col("IS_CURRENT"))
    assert_unique(current, keys, f"{dimension_name} current")
    current_lookup = current.select(*keys, F.col("RECORD_HASH").alias("_TARGET_HASH"),
                                    F.col("DIMENSION_SK").alias("_CURRENT_SK"), F.col("EFFECTIVE_FROM").alias("_CURRENT_FROM"))
    historical_keys = history.select(*keys).distinct().withColumn("_HAS_HISTORY", F.lit(True))
    classified = (staging.join(current_lookup, keys, "left").join(historical_keys, keys, "left")
                  .withColumn("_CHANGE", F.when(F.col("_CURRENT_SK").isNull(), "INSERT")
                              .when(~F.col("RECORD_HASH").eqNullSafe(F.col("_TARGET_HASH")), "UPDATE").otherwise("NO_CHANGE")))
    inserts = classified.filter(F.col("_CHANGE") == "INSERT").count()
    changes = classified.filter(F.col("_CHANGE") == "UPDATE").count()
    unchanged = classified.filter(F.col("_CHANGE") == "NO_CHANGE").count()
    initial_default = dimension_version_start(None, False, change_date, effective_dt.strip())
    returning_date = dimension_version_start(None, True, change_date, effective_dt.strip())
    initial_date = (F.col("DATAB") if "DATAB" in business_columns and not effective_dt.strip() else F.lit(initial_default))
    versions = (classified.filter(F.col("_CHANGE") != "NO_CHANGE")
                .withColumn("EFFECTIVE_FROM", F.when(F.col("_CURRENT_SK").isNotNull(), F.greatest(F.lit(change_date), F.col("_CURRENT_FROM")))
                            .when(F.col("_HAS_HISTORY"), F.lit(returning_date)).otherwise(initial_date)))
    # Reuse an existing daily version (including pre-refactor SKs) on corrections/reappearance.
    version_lookup = history.select(*keys, "EFFECTIVE_FROM", F.col("DIMENSION_SK").alias("_EXISTING_SK"))
    versions = versions.join(version_lookup, keys + ["EFFECTIVE_FROM"], "left")
    versions = (versions.withColumn("EFFECTIVE_TO", F.lit(HIGH_DATE).cast("date")).withColumn("IS_CURRENT", F.lit(True))
                .withColumn("DIMENSION_SK", F.coalesce(F.col("_EXISTING_SK"), F.md5(F.concat_ws("||", *[
                    F.coalesce(F.col(key).cast("string"), F.lit("__NULL__")) for key in keys
                ], F.col("EFFECTIVE_FROM").cast("string")))))
                .withColumn("GOLD_BATCH_ID", F.lit(BATCH_ID)).withColumn("GOLD_LOAD_TIMESTAMP", F.lit(RUN_TIMESTAMP)))
    to_expire = versions.filter(F.col("_CURRENT_SK").isNotNull() & (F.col("EFFECTIVE_FROM") > F.col("_CURRENT_FROM"))).select(
        F.col("_CURRENT_SK").alias("DIMENSION_SK"), F.date_sub("EFFECTIVE_FROM", 1).alias("_END")
    )
    expirations = expire_versions(current.join(to_expire, "DIMENSION_SK", "inner"), F.col("_END")).drop("_END")
    # Complete Silver state is authoritative for small masters, including an empty source.
    removed = expire_versions(current.join(staging.select(*keys), keys, "left_anti"), F.date_sub(F.lit(change_date), 1))
    actions = versions.select(*history.columns).unionByName(expirations).unionByName(removed)
    assert_unique(actions, ["DIMENSION_SK"], f"{dimension_name} actions")
    if actions.limit(1).count():
        (DeltaTable.forName(spark, target).alias("tgt").merge(actions.alias("src"), "tgt.DIMENSION_SK = src.DIMENSION_SK")
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    return (dimension_name, staging.count(), inserts, changes, unchanged, "SUCCESS")


# Conformed G/L account grain is unchanged; English texts can independently change attributes.
ska1 = latest_per_key(spark.table(f"{SLV}.ska1"), ["MANDT", "KTOPL", "SAKNR"]).alias("a")
skat = latest_per_key(spark.table(f"{SLV}.skat"), ["MANDT", "SPRAS", "KTOPL", "SAKNR"]).filter(F.col("SPRAS") == "E").alias("t")
gl_keys = ["MANDT", "KTOPL", "SAKNR"]
gl = ska1.join(skat, gl_keys, "left").select(*[F.col(f"a.{column}").alias(column) for column in ska1.columns],
                                           *[F.col(f"t.{column}").alias(column) for column in ["TXT20", "TXT50"] if column in skat.columns])
gl.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{SLV}.gl_account_conformed")

# Division remains one conformed row per KHINR, with deterministic label and validity bounds.
cepc_latest = latest_per_key(spark.table(f"{SLV}.cepc"), ["MANDT", "KOKRS", "PRCTR", "DATBI"])
division_rows = cepc_latest.select(F.col("KHINR").alias("DIVISION_CODE"), F.col("SEGMENT").alias("DIVISION_NAME"),
                                  "DATAB", "DATBI", *SOURCE_METADATA).filter(F.col("DIVISION_CODE").isNotNull())
division_bounds = division_rows.groupBy("DIVISION_CODE").agg(F.min("DATAB").alias("DATAB"), F.max("DATBI").alias("DATBI"))
label_order = Window.partitionBy("DIVISION_CODE").orderBy(F.col("DATAB").desc(), F.col("DATBI").desc(), F.col("DIVISION_NAME").asc_nulls_last())
division = (division_rows.withColumn("_RN", F.row_number().over(label_order)).filter(F.col("_RN") == 1)
            .drop("_RN", "DATAB", "DATBI").join(division_bounds, "DIVISION_CODE"))
division.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{SLV}.division_conformed")

results = [scd2_load(name, cfg) for name, cfg in DIM_CONFIG.items()]
run_log = spark.createDataFrame(results, "dimension_name string, staged_rows long, inserted_keys long, changed_keys long, unchanged_keys long, status string").withColumn("gold_batch_id", F.lit(BATCH_ID)).withColumn("gold_load_timestamp", F.lit(RUN_TIMESTAMP))
run_log.write.mode("append").format("delta").saveAsTable(f"{AUD}.gold_scd2_run_log")
display(run_log)

# Date dimension. Calendar fiscal mapping is a demo assumption and must be replaced with Eskom's confirmed fiscal calendar.
dates=spark.sql("SELECT explode(sequence(to_date('2018-01-01'),to_date('2035-12-31'),interval 1 day)) AS CALENDAR_DATE")
dates=(dates.withColumn("DATE_SK",F.date_format("CALENDAR_DATE","yyyyMMdd").cast("int")).withColumn("CALENDAR_YEAR",F.year("CALENDAR_DATE")).withColumn("QUARTER_NUMBER",F.quarter("CALENDAR_DATE")).withColumn("MONTH_NUMBER",F.month("CALENDAR_DATE")).withColumn("MONTH_NAME",F.date_format("CALENDAR_DATE","MMMM")).withColumn("FISCAL_YEAR",F.year("CALENDAR_DATE")).withColumn("FISCAL_PERIOD",F.month("CALENDAR_DATE")))
dates.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{DIM}.date")

# Fiscal period dimension supports SAP carry-forward 000, normal 1-12, and special 13-16.
fiscal_years = spark.range(2018, 2036).select(
    F.col("id").cast("int").alias("FISCAL_YEAR")
)
fiscal_period_numbers = spark.range(0, 17).select(
    F.col("id").cast("int").alias("FISCAL_PERIOD")
)
fiscal_periods = (
    fiscal_years.crossJoin(fiscal_period_numbers)
    .withColumn(
        "FISCAL_PERIOD_KEY",
        (F.col("FISCAL_YEAR") * F.lit(100) + F.col("FISCAL_PERIOD")).cast("int"),
    )
    .withColumn(
        "FISCAL_PERIOD_NAME",
        F.when(F.col("FISCAL_PERIOD") == 0, F.lit("Carry Forward")).when(
            F.col("FISCAL_PERIOD") <= 12,
            F.concat(F.lit("Period "), F.lpad(F.col("FISCAL_PERIOD"), 2, "0")),
        ).otherwise(
            F.concat(
                F.lit("Special Period "),
                F.lpad(F.col("FISCAL_PERIOD"), 2, "0"),
            )
        ),
    )
    .withColumn(
        "FISCAL_QUARTER",
        F.when(F.col("FISCAL_PERIOD") == 0, F.lit("Carry Forward")).when(
            F.col("FISCAL_PERIOD") <= 12,
            F.concat(
                F.lit("Q"),
                F.ceil(F.col("FISCAL_PERIOD") / F.lit(3)).cast("int").cast("string"),
            ),
        ).otherwise(F.lit("Special")),
    )
    .withColumn("IS_SPECIAL_PERIOD", F.col("FISCAL_PERIOD") > 12)
    .withColumn(
        "FISCAL_YEAR_PERIOD_LABEL",
        F.concat(
            F.col("FISCAL_YEAR").cast("string"),
            F.lit("-P"),
            F.lpad(F.col("FISCAL_PERIOD"), 2, "0"),
        ),
    )
    .withColumn("FISCAL_YEAR_PERIOD_SORT", F.col("FISCAL_PERIOD_KEY"))
    .select(
        "FISCAL_PERIOD_KEY",
        "FISCAL_YEAR",
        "FISCAL_PERIOD",
        "FISCAL_PERIOD_NAME",
        "FISCAL_QUARTER",
        "IS_SPECIAL_PERIOD",
        "FISCAL_YEAR_PERIOD_LABEL",
        "FISCAL_YEAR_PERIOD_SORT",
    )
)
fiscal_periods.write.mode("overwrite").option(
    "overwriteSchema", "true"
).format("delta").saveAsTable(f"{DIM}.fiscal_period")
